1. *Data Description*

Dataset Overview:

(1) The dataset consists of two related tables: a player dataset and a session dataset.

(2) The player dataset contains 196 unique players and 7 variables describing demographics and play history.

(3) The session dataset contains 1535 recorded play sessions and 5 variables detailing session start/end times


data collection method: the data is collected passively from a Minecraft server. `players.csv` contains registration and profile information provided by users upon signing up. `sessions.csv` is an automated log of player connection and disconnection times, generated by the server software.

dataset 1: player demographics and profile
| Variable     | Type        | Description                                       |
| ------------ | ----------- | ------------------------------------------------- |
| experience   | factor      | Self-reported experience level                    |
| subscribe    | logical     | Indicates whether player subscribed to newsletter |
| hashedEmail  | character   | Anonymous unique player ID                        |
| played_hours | double      | Total hours spent playing                         |
| name         | character   | Player display name                               |
| gender       | factor      | Reported gender                                   |
| Age          | integer     | Age in years                                      |

dataset 2: player session logs
| Variable Name           | Type                      | Meaning                                                                                                                                                             |
| ----------------------- | ------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **hashedEmail**         | character                    | A hashed (anonymized) unique identifier for each player. This is used to link sessions to the player dataset.                                                       |
| **start_time**          | character         | The recorded start time of the session, formatted as `DD/MM/YYYY HH:MM`. Requires conversion to a proper datetime type for analysis.                                |
| **end_time**            | character         | The recorded end time of the session, also formatted as `DD/MM/YYYY HH:MM`. Paired with `start_time` to compute session duration.                                   |
| **original_start_time** | double | Start time recorded as a Unix-like timestamp in scientific notation (e.g., `1.71977E+12`). This provides a more precise, timezone-standardized way to measure time. |
| **original_end_time**   | double | End time recorded in the same epoch timestamp format as `original_start_time`. Useful for precise duration calculations and time-series modeling.                   |


In [ ]:
library(tidyverse)
library(repr)
library(tidymodels)
library(GGally)
library(ISLR)
options(repr.matrix.max.rows = 6)
options(scipen = 999)

*2. Questions*

Broad Question：

Question 3: We are interested in demand forecasting, namely, what time windows are most likely to have large number of simultaneous players. This is because we need to ensure that the number of licenses on hand is sufficiently large to accommodate all parallel players with high probability.


Specific Question：

“Can the total number of hours a player has accumulated (from players.csv) and the duration of their previous sessions (from sessions.csv) predict whether they will start a new session in the next 24 hours?”

Predictive method:

K-nearest neighbors classification (only one method)


How the data will help address the question：

To address this question, I will combine information from `players.csv` and `sessions.csv` using `hashedEmail`. From these data, I will create predictors such as a player’s total hours played and their average session duration. I will also compute a binary response variable that indicates whether a player begins another session within 24 hours of their most recent one. Using tidyverse tools like `mutate`, `summarize`, and `group_by`, I will construct a tidy dataset, which will then be used to train a K-nearest neighbors classification model to predict whether a player will return within 24 hours.

In [ ]:
players <- read.csv("players.csv")
sessions <- read.csv("sessions.csv")

sessions <- sessions |>
  mutate(
    start_time = as.POSIXct(start_time, format = "%d/%m/%Y %H:%M"),
    end_time   = as.POSIXct(end_time, format = "%d/%m/%Y %H:%M")
  ) |>
  select(-original_start_time, -original_end_time)
sessions
players

players_means <- players |>
  select(where(is.numeric)) |>
  summarize(across(everything(), mean, na.rm = TRUE))

players_means

played_hours_plot <- players |>
  ggplot(aes(x = played_hours)) +
  geom_histogram(bins = 20, fill = "grey70") +
  labs(
    title = "Distribution of Total Played Hours",
    x = "Played Hours",
    y = "Number of Players"
  )
played_hours_plot

age_plot <- players |>
  ggplot(aes(x = Age)) +
  geom_histogram(bins = 20, fill = "grey70") +
  labs(
    title = "Distribution of Player Ages",
    x = "Age (years)",
    y = "Number of Players"
  )
age_plot

sessions <- sessions |>
  mutate(duration_minutes = as.numeric(difftime(end_time, start_time, units = "mins")))

session_plot <- sessions |>
  ggplot(aes(x = duration_minutes)) +
  geom_histogram(bins = 20) +
  labs(
    title = "Distribution of Session Duration",
    x = "Duration (minutes)",
    y = "Number of Sessions"
  )
session_plot

Overall insights:

Played hours are heavily skewed, indicating large differences in engagement that may require scaling later.

Age distribution is concentrated in younger users, especially those around 20, which might influence return behavior.

The histogram of session duration shows that most sessions are short, with a long right tail indicating a small number of much longer sessions, suggesting substantial variability in how long players stay in the game.

*(4) Methods and Plan*
Proposed Method: K-Nearest Neighbors (KNN) Classification


To address the research question
“Which player characteristics best predict whether a player will subscribe to the game?”,
I will use a K-Nearest Neighbors (KNN) classification model. The outcome variable is the binary variable subscribe, and the predictors include player-level attributes such as age, experience level, played hours, average session duration, and number of sessions.


Why is this method appropriate?

KNN is a classification method covered in the course and is suitable for binary outcomes.

The method is intuitive: players are predicted to subscribe (or not) based on the subscription behavior of their most similar neighbors.

KNN does not require assumptions about linearity or distribution, making it flexible for this type of dataset.

It works with both numerical and categorical variables once they are preprocessed appropriately.



Required Assumptions:

Similarity assumption: players who are close in feature space should have similar outcomes.

Meaningful distance measure: the distance metric should reflect meaningful differences between players.

Standardized features: numeric predictors must be scaled so that no variable dominates the distance.

Adequate data density: the dataset must contain enough nearby observations for the model to make reliable decisions.



Potential Limitations:

KNN can perform poorly in high-dimensional settings.

It is sensitive to outliers, which can distort distances.

Choosing an appropriate k is crucial and requires tuning.

Compared with other models, KNN offers limited interpretability.



Model Comparison and Selection:

To choose the best KNN model, I will use:

10-fold cross-validation on the training set

Cross-validation will help:

Evaluate performance

Compare different values of k (e.g., 3, 5, 7, 9, …)

Select the k that produces the highest accuracy

Because ROC AUC is not part of the course, evaluation will be based on:

Accuracy (main metric)

Confusion matrix if needed

The final model will use the k with the best cross-validated accuracy.

Data Processing Plan
1. Data Wrangling

I will:

Merge the player dataset and session dataset using hashedEmail.

Construct player-level features such as average session duration and the number of sessions.

Ensure the resulting dataset is tidy.

2. Train/Test Split

To avoid data leakage, the split will be done before any preprocessing:

80% training set

20% test set

Only the training data will be used to build the preprocessing steps and model.

3. Cross-Validation

Within the training data:

Perform 10-fold cross-validation

Try multiple k values

Select the k that results in the highest cross-validated accuracy

4. Feature Processing

Because KNN relies on distance, appropriate preprocessing is required:

Standardize all numeric variables (centering and scaling).

Convert categorical variables into numerical indicators in a simple and consistent way (using whichever method was taught in the course).
(No specific function names are mentioned to keep it within course scope.)

Apply the preprocessing steps only on the training data.